In [4]:
# ============================================================================
# NB03b_data_dictionary_check.ipynb  —  Nested CV Version, Step 1: fold 0
# (Re-run against NB03's updated output: 22 candidates after VIF
#  threshold was lowered from 10 to 5 — see NB03 fold 0 log for rationale.
#  No logic changes here; duplicate-correlation threshold remains 0.90
#  with union-find grouping, as established previously.)
#
# Study 2: Verify variable definitions and detect duplicate/redundant features
#          before association rule mining (NB04)
#
# Input  : data/processed/fold_0/upgrade_cohort_fold0.parquet
#           -> filtered to outer_split == "train"
#          results/tables/NB03_02_top_candidates_fold0.csv  (now 22 rows)
# Output : results/tables/NB03b_variable_map_fold0.csv — final deduplicated variable list
# ============================================================================

import os
import numpy as np
import pandas as pd

PROC_DIR  = "../data/processed/"
TABLE_DIR = "../results/tables/"

RANDOM_SEED = 42
FOLD_ID     = 0

DUPLICATE_CORR_THRESHOLD = 0.90

# ── Load fold 0 train partition + NB03's VIF-screened candidates ────────────

fold_cohort_path = os.path.join(PROC_DIR, f"fold_{FOLD_ID}", f"upgrade_cohort_fold{FOLD_ID}.parquet")
cohort_full_fold = pd.read_parquet(fold_cohort_path)
cohort = cohort_full_fold[cohort_full_fold["outer_split"] == "train"].copy()

candidates_path = os.path.join(TABLE_DIR, f"NB03_02_top_candidates_fold{FOLD_ID}.csv")
candidates = pd.read_csv(candidates_path, index_col=0)

print(f"Loaded fold {FOLD_ID} train partition: {cohort.shape}")
print(f"Candidate variables from NB03 (fold {FOLD_ID}) : {len(candidates)}")
print(f"  [VIF threshold=5 was used to produce this list — see NB03 log]")
print(candidates.index.tolist())


# ── Known variable name map (from UCI dataset documentation) ──────────────────

ATTR_MAP = {
    "Attr1" : "net profit / total assets",
    "Attr2" : "total liabilities / total assets",
    "Attr3" : "working capital / total assets",
    "Attr4" : "current assets / short-term liabilities",
    "Attr5" : "cash / current liabilities",
    "Attr6" : "retained earnings / total assets",
    "Attr7" : "EBIT / total assets",
    "Attr8" : "book value of equity / total liabilities",
    "Attr9" : "sales / total assets",
    "Attr10": "equity / total assets",
    "Attr11": "gross profit + extraordinary items + financial expenses / total assets",
    "Attr12": "gross profit / short-term liabilities",
    "Attr13": "gross profit + depreciation / sales",
    "Attr14": "gross profit + interest / total assets",
    "Attr15": "total liabilities × 365 / (gross profit + depreciation)",
    "Attr16": "gross profit / total assets",
    "Attr17": "total assets / total liabilities",
    "Attr18": "gross profit / total assets (alt)",
    "Attr19": "gross profit / sales",
    "Attr20": "inventory × 365 / sales",
    "Attr21": "sales (t) / sales (t-1)",
    "Attr22": "profit on operating activities / total assets",
    "Attr23": "net profit / sales",
    "Attr24": "gross profit (3-year avg) / total assets",
    "Attr25": "equity / total assets (alt)",
    "Attr26": "gross profit / (depreciation × total assets)",
    "Attr27": "profit on operating activities / financial expenses",
    "Attr28": "working capital / fixed assets",
    "Attr29": "log(total assets)",
    "Attr30": "total liabilities / cash",
    "Attr31": "(short-term liabilities × 365) / COGS",
    "Attr32": "operating expenses / short-term liabilities",
    "Attr33": "operating expenses / total liabilities",
    "Attr34": "profit on sales / total assets",
    "Attr35": "total sales / total assets (alt)",
    "Attr36": "total sales / total assets",
    "Attr37": "(current assets - inventory - short-term receivables) / short-term liabilities",
    "Attr38": "total liabilities / total assets (alt)",
    "Attr39": "net profit / total assets (alt)",
    "Attr40": "total assets / total liabilities (alt)",
    "Attr41": "short-term liabilities / total assets",
    "Attr42": "(short-term liabilities + long-term liabilities) / equity",
    "Attr43": "profit on operating activities / depreciation",
    "Attr44": "total liabilities / (profit + depreciation)",
    "Attr45": "profit on sales / sales",
    "Attr46": "(current assets - inventory) / short-term liabilities",
    "Attr47": "(current assets - short-term liabilities) / (current assets - inventory - short-term liabilities)",
    "Attr48": "total liabilities / total assets (alt 2)",
    "Attr49": "(net profit + depreciation) / total liabilities",
    "Attr50": "total expenses / total sales",
    "Attr51": "total liabilities / (sales + other operating income)",
    "Attr52": "short-term liabilities / sales",
    "Attr53": "short-term liabilities / total assets",
    "Attr54": "total liabilities / (net profit + depreciation)",
    "Attr55": "profit on operating activities (×1000) / short-term liabilities",
    "Attr56": "net profit / total assets - ROA",
    "Attr57": "total assets / current liabilities",
    "Attr58": "total costs / total sales",
    "Attr59": "long-term liabilities / equity",
    "Attr60": "sales / inventory",
    "Attr61": "sales / receivables",
    "Attr62": "short-term liabilities × 365 / sales",
    "Attr63": "sales / short-term liabilities",
    "Attr64": "sales / fixed assets",
}

NAMED_MAP = {
    "net_profit_to_assets"               : "Attr1  — net profit / total assets",
    "total_liabilities_to_assets"        : "Attr2  — total liabilities / total assets",
    "working_capital_to_assets"          : "Attr3  — working capital / total assets",
    "current_assets_to_short_liabilities": "Attr4  — current assets / short-term liabilities",
    "cash_to_current_liabilities"        : "Attr5  — cash / current liabilities",
    "retained_earnings_to_assets"        : "Attr6  — retained earnings / total assets",
    "ebit_to_assets"                     : "Attr7  — EBIT / total assets",
    "book_value_equity_to_liabilities"   : "Attr8  — book value of equity / total liabilities",
    "sales_to_assets"                    : "Attr9  — sales / total assets",
    "equity_to_assets"                   : "Attr10 — equity / total assets",
}


# ── Step 1: Print definition of each candidate variable ───────────────────────

print("\n" + "=" * 70)
print(f"CANDIDATE VARIABLE DEFINITIONS — FOLD {FOLD_ID}")
print("=" * 70)

rows = []
for feat in candidates.index:
    if feat in NAMED_MAP:
        definition = NAMED_MAP[feat]
    elif feat in ATTR_MAP:
        definition = ATTR_MAP[feat]
    else:
        definition = "— unknown —"
    rows.append({"feature": feat, "definition": definition})
    print(f"  {feat:<35s} {definition}")

def_df = pd.DataFrame(rows).set_index("feature")


# ── Step 2: Correlation-based duplicate detection (union-find grouping) ──────

print("\n" + "=" * 70)
print(f"CORRELATION-BASED DUPLICATE DETECTION — FOLD {FOLD_ID}  "
      f"(|r| > {DUPLICATE_CORR_THRESHOLD}, union-find grouping)")
print("=" * 70)

cand_cols   = candidates.index.tolist()
corr_matrix = cohort[cand_cols].corr().abs()

parent = {v: v for v in cand_cols}

def find(v):
    while parent[v] != v:
        v = parent[v]
    return v

def union(v1, v2):
    r1, r2 = find(v1), find(v2)
    if r1 != r2:
        parent[r2] = r1

duplicate_pairs_found = []
for i, c1 in enumerate(cand_cols):
    for c2 in cand_cols[i+1:]:
        r = corr_matrix.loc[c1, c2]
        if r > DUPLICATE_CORR_THRESHOLD:
            duplicate_pairs_found.append((c1, c2, round(r, 4)))
            union(c1, c2)
            print(f"  |r|={r:.4f}  {c1}  ↔  {c2}")
            print(f"         {ATTR_MAP.get(c1, NAMED_MAP.get(c1, '?'))}")
            print(f"         {ATTR_MAP.get(c2, NAMED_MAP.get(c2, '?'))}")

if not duplicate_pairs_found:
    print(f"  No near-duplicate pairs found (|r| > {DUPLICATE_CORR_THRESHOLD}).")


# ── Step 3: Select one representative per duplicate GROUP ────────────────────

groups = {}
for v in cand_cols:
    root = find(v)
    groups.setdefault(root, []).append(v)

to_drop = set()
print(f"\nDuplicate groups (size > 1):")
found_any_group = False
for root, members in groups.items():
    if len(members) > 1:
        found_any_group = True
        rb_abs = candidates.loc[members, "rank_biserial"].abs()
        keep_var = rb_abs.idxmax()
        drop_vars = [m for m in members if m != keep_var]
        to_drop.update(drop_vars)
        print(f"  Group {members}")
        print(f"    Keep: {keep_var} (|rb|={rb_abs[keep_var]:.4f})  "
              f"→  Drop: {drop_vars}")
if not found_any_group:
    print("  (none)")

final_candidates = candidates.drop(index=list(to_drop))

print(f"\nVariables dropped as duplicates : {len(to_drop)}")
print(f"Final candidate list            : {len(final_candidates)}")


# ── Step 4: Final variable list with definitions ──────────────────────────────

print("\n" + "=" * 70)
print(f"FINAL VARIABLE LIST FOR NB04 — FOLD {FOLD_ID}")
print("=" * 70)

final_df = def_df.loc[final_candidates.index].copy()
final_df["OR_mle"]        = candidates.loc[final_candidates.index, "OR_mle"]
final_df["rank_biserial"] = candidates.loc[final_candidates.index, "rank_biserial"]
final_df["direction"]     = candidates.loc[final_candidates.index, "direction"]

for feat, row in final_df.iterrows():
    print(f"  {feat:<35s} OR={row['OR_mle']:.4f}  rb={row['rank_biserial']:.4f}"
          f"  [{row['direction']}]")
    print(f"    → {row['definition']}")

out_path = os.path.join(TABLE_DIR, f"NB03b_variable_map_fold{FOLD_ID}.csv")
final_df.to_csv(out_path)
print(f"\nSaved: {out_path}")

print("\n" + "=" * 60)
print(f"NB03b SUMMARY — FOLD {FOLD_ID}")
print("=" * 60)
print(f"Candidates in (from NB03 VIF screening, threshold=5) : {len(candidates)}")
print(f"Duplicate-correlation threshold used                  : {DUPLICATE_CORR_THRESHOLD}")
print(f"Dropped as near-duplicates                             : {len(to_drop)}")
print(f"Final variable list for NB04                           : {len(final_df)}")
print()
print(f"Next step → NB04_association_rules.ipynb (re-run with this "
      f"{len(final_df)}-variable list)")

Loaded fold 0 train partition: (12537, 81)
Candidate variables from NB03 (fold 0) : 22
  [VIF threshold=5 was used to produce this list — see NB03 log]
['Attr52', 'Attr49', 'Attr38', 'working_capital_to_assets', 'Attr30', 'Attr25', 'Attr13', 'Attr24', 'Attr35', 'retained_earnings_to_assets', 'Attr55', 'Attr41', 'Attr29', 'Attr56', 'Attr21', 'Attr12', 'cash_to_current_liabilities', 'Attr57', 'Attr63', 'Attr53', 'Attr58', 'Attr46']

CANDIDATE VARIABLE DEFINITIONS — FOLD 0
  Attr52                              short-term liabilities / sales
  Attr49                              (net profit + depreciation) / total liabilities
  Attr38                              total liabilities / total assets (alt)
  working_capital_to_assets           Attr3  — working capital / total assets
  Attr30                              total liabilities / cash
  Attr25                              equity / total assets (alt)
  Attr13                              gross profit + depreciation / sales
  Attr24    